# AIDM · Modelo de decisión (v2): estado de la respuesta, uso de herramientas y despliegue

Este cuaderno entrena **un solo modelo** que contesta tres preguntas distintas de AIDM. El modelo no escribe texto: hace una sola pasada y devuelve una probabilidad por opción, así que se sirve en CPU, con llama.cpp, en el mismo ordenador que AIDM (ver `modelo_decision/README.md`). Las preguntas y los textos que lee son los de `decisiones.py`, los mismos que usa AIDM.

**1. `estado` (fallos 36 y 68).** En cada turno del discovery: qué hace la respuesta del cliente con el campo por el que se le ha preguntado.

| Opción | Qué significa | Qué hace AIDM |
|---|---|---|
| `dato` | da el dato, aunque sea aproximado o negativo («no tenemos») | nada: el valor lo guarda el LLM |
| `no_lo_sabe` | dice que no lo sabe | guarda «Confirmado por el usuario: no lo sabe» |
| `no_aplica` | dice que no aplica a su proyecto | guarda «Confirmado por el usuario: no aplica» |
| `no_responde` | no contesta a lo preguntado | nada: el campo sigue pendiente |
| `vaga` | habla del dato sin concretar («bastantes», «lo normal») | no lo guarda y vuelve a preguntar (una vez) |

**2. `uso` (fallo 69).** En el agente de plataforma, para cada herramienta o nube que el cliente nombra en un campo con una negación o un cambio («no usan Power BI», «dejaron Oracle»): `la_usa`, `quiere_usarla`, `no_la_usa` o `la_deja`. Las dos últimas no suman a esa nube ni a su ecosistema. Una fila puede llevar hasta 4 preguntas (`uso_1`…`uso_4`), una por herramienta.

**3. `despliegue` (fallo 70).** Con los datos del caso: `cloud`, `hibrido`, `on_premises` o `sin_datos`. Es decisión de Inetum, no una preferencia del cliente; la regla con la que se han etiquetado los ejemplos está en `despliegue_correcto()` de `generar_datos.py`.

**Pasos:** instalar → datos → modelo base → entrenar → evaluar por tarea → probar a mano → probar en CPU con llama.cpp → subir el GGUF a Hugging Face.

**Antes de empezar:**
1. Sube a la rama los cambios de los fallos 68, 69 y 70 y los datos nuevos (`modelo_decision/datos/`): el cuaderno los baja de ahí, junto con `decisiones.py`.
2. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
3. En la llave de la izquierda (*Secretos*), `HF_TOKEN` con un token de Hugging Face **de escritura** y acceso al cuaderno.
4. Tu usuario de Hugging Face en la celda de configuración.

Los ejemplos son inventados (`modelo_decision/generar_datos.py`): no hay datos de clientes.

## 1. Instalar Unsloth
Es la celda del cuaderno oficial de Unsloth. Tarda unos minutos.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Configuración

In [ ]:
USUARIO_HF = "tu_usuario"            # tu usuario de Hugging Face
REPO_MODELO = "aidm-decision"        # repo privado donde se sube el GGUF (la v1 queda en su historial)
MODELO_BASE = "unsloth/Qwen3.5-0.8B"
RAMA = "https://raw.githubusercontent.com/ruBBBBBBa/AIDM/pruebas-rubengs"  # de aquí se bajan los datos y decisiones.py
UMBRAL = 0.7                         # la misma confianza mínima que decisiones.CONFIANZA_MINIMA en AIDM
TAREAS = ("estado", "uso", "despliegue")
MUESTRA_CPU = 40                     # ejemplos de prueba por tarea para la prueba en CPU del paso 8 (10 s cada uno)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert USUARIO_HF != "tu_usuario", "Pon tu usuario de Hugging Face en USUARIO_HF"

## 3. Datos
Se bajan de la rama, con `decisiones.py` (las preguntas y los textos que usa AIDM). Si todavía no los has subido, súbelos a mano desde el panel de archivos: los dos `.jsonl` de `modelo_decision/datos/` a la carpeta `datos/`, y `decisiones.py` a la raíz.

In [ ]:
import json, os, urllib.request
from collections import Counter

os.makedirs("datos", exist_ok=True)
for ruta, origen in (("datos/entrenamiento.jsonl", "modelo_decision/datos/entrenamiento.jsonl"),
                     ("datos/prueba.jsonl", "modelo_decision/datos/prueba.jsonl"),
                     ("decisiones.py", "decisiones.py")):
    if not os.path.exists(ruta):
        try:
            urllib.request.urlretrieve(f"{RAMA}/{origen}", ruta)
        except Exception as exc:
            print(f"No se pudo bajar {origen} ({exc}): súbelo a mano")

def leer(ruta):
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(linea) for linea in f if linea.strip()]

entrenamiento, prueba = leer("datos/entrenamiento.jsonl"), leer("datos/prueba.jsonl")
faltan = set(TAREAS) - {f.get("tarea") for f in entrenamiento}
assert not faltan, (f"Los datos no traen {sorted(faltan)}: son los de antes. Bórralos de datos/ y vuelve a ejecutar "
                    "esta celda después de subir a la rama los nuevos (python modelo_decision/generar_datos.py).")

for nombre, filas in (("entrenamiento", entrenamiento), ("prueba", prueba)):
    print(f"{nombre}: {len(filas)} filas")
    for tarea in TAREAS:
        reparto = Counter(v for f in filas if f["tarea"] == tarea for v in f["gold"].values())
        print(f"  {tarea}: {dict(sorted(reparto.items()))}")
for tarea in TAREAS:
    f = next(f for f in entrenamiento if f["tarea"] == tarea)
    print(f"\n[{tarea}]\n{f['state']}\n→ {f['gold']}")

## 4. Modelo base
Qwen3.5-0.8B en 4 bits con LoRA (r=16), como recomienda la guía de Unsloth. La cabeza de decisión es nueva y siempre se entrena. Las tres tareas van mezcladas en el mismo entrenamiento: la cabeza lee cada pregunta con sus opciones, así que un solo modelo sirve para las tres.

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = MODELO_BASE,
    max_seq_length = 1024,   # la fila más larga (un caso de despliegue, o 4 preguntas de uso) ocupa unos 600 tokens
    load_in_4bit = True,
)
model = FastDecisionModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
items, informe = FastDecisionModel.build_dataset(entrenamiento, tokenizer, model)
print(f"Entrenamiento: descartados {informe['skipped']} de {informe['total']}", informe["reason"] or "")
train_items, eval_items = FastDecisionModel.split_holdout(items, seed = 3407)  # para calibrar
test_items, informe_prueba = FastDecisionModel.build_dataset(prueba, tokenizer, model)
print(f"{len(train_items)} para entrenar, {len(eval_items)} apartados para calibrar, {len(test_items)} de prueba")

Antes de entrenar, la cabeza es nueva y acierta más o menos por azar:

In [ ]:
FastDecisionModel.evaluate(model, tokenizer, test_items)

## 5. Entrenar
Dos épocas, como en la guía. En una T4, Qwen3.5 entrena en float32 (la T4 no tiene bfloat16), así que va lento. Hay unas 4.300 filas, el doble que en la v1, así que tarda más o menos el doble.

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = eval_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min de entrenamiento")

**Opcional:** guarda los adaptadores en un repo privado. Si Colab corta la sesión, puedes cargarlos y exportar sin volver a entrenar (`FastDecisionModel.from_pretrained(f"{USUARIO_HF}/{REPO_MODELO}-lora", load_in_4bit=True)`).

In [ ]:
model.push_to_hub(f"{USUARIO_HF}/{REPO_MODELO}-lora", tokenizer, token = HF_TOKEN, private = True)

## 6. Calibrar y evaluar
La calibración ajusta las probabilidades para que una confianza de 0,9 acierte más o menos 9 de cada 10 veces. Luego se mide con la prueba, que usa frases que el modelo no ha visto, **tarea por tarea**. Lo que más importa en AIDM es la precisión de lo que hace actuar a AIDM con confianza ≥ 0,7: en `estado`, `no_lo_sabe`, `no_aplica` y `vaga`; en `uso`, `no_la_usa` y `la_deja` (quitan una herramienta); en `despliegue`, cualquier opción menos `sin_datos`.

In [ ]:
print(FastDecisionModel.calibrate(model, tokenizer, eval_items))
print(FastDecisionModel.evaluate(model, tokenizer, test_items))

In [ ]:
import pandas as pd

FastDecisionModel.for_inference(model)
filas = []
for f in prueba:
    respuestas = FastDecisionModel.predict(model, tokenizer, f["state"], f["questions"])
    for clave, real in f["gold"].items():
        r = respuestas[clave]
        confianza = r.get("confidence", (r.get("probabilities") or {}).get(r["answer"], 0.0))
        filas.append({"tarea": f["tarea"], "real": real, "modelo": r["answer"], "confianza": confianza,
                      "idioma": f["idioma"], "pregunta": f["questions"][clave]["instructions"], "estado": f["state"]})
tabla = pd.DataFrame(filas)
tabla["acierta"] = tabla.real == tabla.modelo

# Lo que hace actuar a AIDM en cada tarea (con confianza suficiente)
ACTUAN = {"estado": ["no_lo_sabe", "no_aplica", "vaga"], "uso": ["no_la_usa", "la_deja"],
          "despliegue": ["cloud", "hibrido", "on_premises"]}
for tarea in TAREAS:
    t = tabla[tabla.tarea == tarea]
    print(f"\n===== {tarea}: acierta {t.acierta.mean():.1%} de {len(t)} "
          f"(es {t[t.idioma == 'es'].acierta.mean():.1%}, en {t[t.idioma == 'en'].acierta.mean():.1%})")
    print(pd.crosstab(t.real, t.modelo))
    for opcion in ACTUAN[tarea]:
        marcadas = t[(t.modelo == opcion) & (t.confianza >= UMBRAL)]
        reales = t[t.real == opcion]
        if len(reales):
            print(f"  {opcion}: con confianza ≥ {UMBRAL} marca {len(marcadas)} y acierta "
                  f"{marcadas.acierta.mean() if len(marcadas) else 0:.0%}; recoge {marcadas.acierta.sum()} de {len(reales)}")

tabla[~tabla.acierta][["tarea", "real", "modelo", "confianza", "pregunta", "estado"]]

## 7. Probar a mano
Las preguntas y los textos salen de `decisiones.py` (el de la rama), así que son los mismos que manda AIDM.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "httpx"], check = False)  # las importa decisiones.py
sys.path.insert(0, ".")
import decisiones as D

DESCRIPCIONES = {}
for f in entrenamiento:
    if f["tarea"] == "estado":
        linea = f["state"].split("\n")[0]
        DESCRIPCIONES[f["campo"]] = linea[linea.index(" (") + 2:-1]

def probar(campo, pregunta, respuesta):
    estado = D.texto_estado(campo, DESCRIPCIONES[campo], pregunta, respuesta)
    r = FastDecisionModel.predict(model, tokenizer, estado, {"estado": D.PREGUNTA_ESTADO})["estado"]
    print(f"estado      {r['answer']:<13} {r.get('confidence', 0):.2f}  {campo}: «{respuesta}»")

def probar_uso(campo, texto, nombres):
    preguntas = {f"uso_{i}": D.pregunta_uso(n) for i, n in enumerate(nombres, 1)}
    r = FastDecisionModel.predict(model, tokenizer, D.texto_uso(campo, texto), preguntas)
    print(f"uso         «{texto}»")
    for clave, nombre in zip(preguntas, nombres):
        print(f"              {nombre:<15} {r[clave]['answer']:<13} {r[clave].get('confidence', 0):.2f}")

def probar_despliegue(**datos):
    r = FastDecisionModel.predict(model, tokenizer, D.texto_despliegue(datos),
                                  {"despliegue": D.PREGUNTA_DESPLIEGUE})["despliegue"]
    print(f"despliegue  {r['answer']:<13} {r.get('confidence', 0):.2f}  {datos}")

probar("tipo_de_arquitectura", "¿Tenéis preferencia de arquitectura: data warehouse, data lake o lakehouse?", "No lo sé")
probar("tiene_equipo_datos", "¿Tenéis equipo de datos propio?", "No, no hay equipo de datos")
probar("volumen_de_datos", "¿Qué volumen de datos manejáis?", "Bastantes datos")                # vaga
probar("volumen_de_datos", "¿Qué volumen de datos manejáis?", "Poca cosa, unos 20 GB")          # dato
probar("restriccion_presupuestaria", "¿Tenéis un presupuesto orientativo?", "Lo razonable")    # vaga
probar("herramienta_de_visualizacion", "¿Usáis alguna herramienta de BI?", "Lo normal: Power BI con 5 licencias")  # dato
probar("plazo_objetivo", "When would you need it up and running?", "Fairly soon")              # vaga

probar_uso("stack_tecnologico_actual", "Python y Excel; no usan Power BI ni Microsoft 365", ["Excel", "Power BI", "Microsoft 365"])
probar_uso("cloud_provider", "Estamos dejando AWS; la idea es pasar a Azure", ["AWS", "Azure"])
probar_uso("cloud_provider", "Azure como prioridad, con un módulo actual en AWS (EC2 y S3 para sensores) que no quieren migrar ni rehacer", ["Azure", "AWS"])

probar_despliegue(cloud_provider = "Azure", stack_tecnologico_actual = "Microsoft 365; los SCADA de la planta no pueden ir a la nube")
probar_despliegue(regulacion_y_compliance = "Por contrato con Defensa, ningún dato puede salir de nuestras instalaciones")
probar_despliegue(modelo_de_despliegue = "En local", stack_tecnologico_actual = "Todo es SaaS: Microsoft 365 y Salesforce")
probar_despliegue(sector = "Logística", volumen_de_datos = "Unos 2 TB")                          # sin_datos

## 8. Exportar a GGUF y probarlo en CPU
AIDM usa el modelo con `llama-server` (llama.cpp) en CPU, así que aquí se prueba igual: se baja `llama-server` de llama.cpp, se arranca con 2 hilos y se le hacen las mismas peticiones que hará AIDM a `/v1/systemone`. El puerto es el 8090, porque el 8080 lo ocupa Colab. Con los 2 hilos de Colab tarda unos 10 s por petición, así que se prueba con `MUESTRA_CPU` ejemplos por tarea (unos 20 minutos con 40); en el portátil (i5-10210U, 4 hilos), unos 4 s.

In [ ]:
model.save_pretrained_gguf("exportado", tokenizer, quantization_method = "q8_0", token = HF_TOKEN)
GGUF = "exportado/gguf/model-Q8_0.gguf"
!ls -lh exportado/gguf

In [ ]:
import glob, statistics, subprocess, tarfile, time
import requests

PUERTO = 8090  # el 8080 lo ocupa Colab

# La última versión de llama.cpp a veces aún no tiene sus binarios subidos: se busca en las 10 últimas.
versiones = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10", timeout = 30).json()
archivo = None
for version in versiones if isinstance(versiones, list) else []:
    archivo = next((a for a in version.get("assets", []) if a["name"].endswith("bin-ubuntu-x64.tar.gz")), None)
    if archivo:
        break
if archivo is None:
    print(versiones if isinstance(versiones, dict) else [v["tag_name"] for v in versiones])
    raise RuntimeError("No encuentro el binario de llama.cpp para Linux x64")
print("llama.cpp", version["tag_name"], "·", archivo["name"])
urllib.request.urlretrieve(archivo["browser_download_url"], archivo["name"])
tarfile.open(archivo["name"]).extractall("llama_bin")
servidor = glob.glob("llama_bin/**/llama-server", recursive = True)[0]
os.chmod(servidor, 0o755)

proceso = subprocess.Popen(
    [servidor, "-m", GGUF, "--port", str(PUERTO), "-c", "2048", "-b", "2048", "-ub", "2048", "-t", "2", "--parallel", "1"],
    env = {**os.environ, "LD_LIBRARY_PATH": os.path.dirname(servidor)},
    stdout = open("llama_server.log", "w"), stderr = subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get(f"http://127.0.0.1:{PUERTO}/health", timeout = 2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if proceso.poll() is not None:
        raise RuntimeError("llama-server se ha cerrado: mira llama_server.log")
    time.sleep(2)
else:
    raise RuntimeError("llama-server no arranca: mira llama_server.log")
print("llama-server listo")

In [ ]:
import random

def decidir(estado, preguntas):
    r = requests.post(f"http://127.0.0.1:{PUERTO}/v1/systemone",
                      json = {"state": estado, "questions": preguntas}, timeout = 180)
    r.raise_for_status()
    return r.json()["answers"]

muestra = []
for tarea in TAREAS:
    de_tarea = [f for f in prueba if f["tarea"] == tarea]
    muestra += random.Random(3407).sample(de_tarea, min(MUESTRA_CPU, len(de_tarea)))

tiempos, aciertos, total = [], Counter(), Counter()
for f in muestra:
    inicio = time.time()
    respuestas = decidir(f["state"], f["questions"])
    tiempos.append(time.time() - inicio)
    for clave, real in f["gold"].items():
        total[f["tarea"]] += 1
        aciertos[f["tarea"]] += respuestas[clave]["choice"] == real
for tarea in TAREAS:
    print(f"llama-server en CPU (2 hilos), {tarea}: acierta {aciertos[tarea]} de {total[tarea]} "
          f"({aciertos[tarea] / total[tarea]:.1%})")
print(f"Tiempo por petición: mediana {statistics.median(tiempos):.2f} s, máximo {max(tiempos):.2f} s")
print("Respuesta de ejemplo:", decidir(muestra[0]["state"], muestra[0]["questions"]))

Cada tarea tiene que acertar más o menos lo mismo que en el paso 6 (en la v1, el 6 nov: 89,3 % en el paso 6 y 90,6 % aquí). Si alguna acierta mucho menos, la conversión a GGUF ha fallado: no lo subas.

## 9. Subir el GGUF a Hugging Face (repo privado)

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token = HF_TOKEN)
repo = api.create_repo(f"{USUARIO_HF}/{REPO_MODELO}", private = True, exist_ok = True).repo_id
api.upload_folder(folder_path = "exportado/gguf", repo_id = repo,
                  commit_message = "v2: estado con «vaga» (68), uso de herramientas (69) y despliegue (70)")
print(f"Subido: https://huggingface.co/{repo}  (la v1 sigue en el historial del repo)")
proceso.terminate()

## 10. Siguiente: usarlo desde AIDM
El GGUF nuevo sustituye al de la v1 con el mismo nombre (`model-Q8_0.gguf`), así que en el portátil basta con cambiar el archivo y volver a arrancar `llama-server`, con el mismo comando y el mismo puerto; el `.env` de AIDM no cambia. Los pasos están en `modelo_decision/README.md`. En resumen:
```
llama-server.exe -m model-Q8_0.gguf --port 8090 -c 2048 -b 2048 -ub 2048 -t 4
```
Usa el GGUF nuevo con el código de los fallos 68, 69 y 70: el modelo de la v1 no ha visto las preguntas nuevas.

Servirlo en un Space de Hugging Face (`modelo_decision/space/`) necesita una cuenta PRO: desde 2026, los Spaces Docker son de pago.